# 03 – Bildoptimierung + XMP + lokale Vision-Metadaten

**Status:** Produktion v0.7 · Logik liegt jetzt in `python/bild_optimierung/`
(vorher komplett in diesem Notebook). Dieses Notebook ist nur noch die Steuerung;
dieselben Funktionen nutzt die Oberfläche in `04_pipeline_gui.ipynb`.

Ablauf je Dokument (`bild_lauf.optimiere_dokument`):

1. Docling-JSON + Markdown lesen – **`<BUCH>_korr.md` (Lektorat) ist führend**, `<BUCH>.md` wird mitgezogen.
2. Jedes referenzierte Bild auflösen, auf höchstens **144 ppi** (aus der Docling-Geometrie) verkleinern.
3. Deterministisch klassifizieren und nur **JPEG oder PNG** kodieren (kalibrierte Qualitätsschranken).
4. Optional ein **lokales Vision-Modell** (LM Studio) für Titel, Alt-Text, Beschreibung, Schlagworte befragen –
   mit dem **lektorierten** Text rund um das Bild als Kontext.
5. XMP (Dublin Core, IPTC Core, `docrag`-Namensraum) nativ in JPEG/PNG einbetten und zurücklesen.
6. Bildverweise und Alt-Texte in **beiden** Markdown-Fassungen nachziehen; Docling-JSON aktualisieren
   (URI, Format, Größe, tatsächliche ppi, `meta.description`, `meta.keywords`) und gegen das Docling-Schema prüfen.
7. JSON/CSV-Manifest schreiben, Abschlussprüfung aller Verweise.

Neu gegenüber v0.6: auch Bilder *ohne* Ersparnis bekommen Metadaten; bereits verarbeitete Bilder
(mit docrag-XMP) werden beim erneuten Lauf übernommen statt erneut verlustbehaftet kodiert;
ein fehlerhaftes Bild bricht den Lauf nicht mehr ab.

Referenzen: W3C PNG Third Edition · IPTC Photo Metadata 2025.1 · LM Studio OpenAI-Kompatibilität
und Structured Output · ExifTool XMP/PNG-Tags.

## 00 — Konfiguration

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "python" / "bild_optimierung").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Projekt-Root nicht gefunden: python/bild_optimierung fehlt.")
for p in (PROJECT_ROOT / "python", PROJECT_ROOT / "python" / "pdf_extraction"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import pandas as pd
import pfade
from bild_optimierung.bild_konfig import BildKonfig
from bild_optimierung import bild_lauf
from pipeline.arbeitsbereich import lektorat_zustand

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

# --- Anzupassen
BUCH = None                                  # None: automatisch aus data/raw
LMS = "http://192.168.178.27:1234/v1"
MODELL = "google/gemma-4-12b"

gefunden = sorted(p.stem for p in pfade.RAW.glob("*") if p.suffix.lower() == ".pdf") if pfade.RAW.exists() else []
if BUCH is None:
    if not gefunden:
        raise RuntimeError("Kein PDF unter data/raw gefunden.")
    BUCH = gefunden[0]
    if len(gefunden) > 1:
        print(f"! {len(gefunden)} PDFs gefunden – BUCH ggf. von Hand setzen. Vorläufig: {BUCH!r}")

DOCUMENT_DIR = pfade.dokument_ordner(BUCH)
for label, pfad in [("Dokumentordner", DOCUMENT_DIR), ("Docling JSON", pfade.dokument_json(BUCH)),
                    ("Markdown (lektoriert)", pfade.dokument_korr_md(BUCH)),
                    ("Markdown (OCR)", pfade.dokument_md(BUCH)),
                    ("Artifacts", pfade.artefakt_ordner(BUCH))]:
    print(f"{label:22s}: {pfad} -> {'vorhanden' if pfad.exists() else 'FEHLT'}")
print("Lektorat               :", lektorat_zustand(BUCH))

CFG = BildKonfig(
    lm_base_url=LMS,
    lm_model=MODELL,
    enable_llm=True,
    max_ppi=144,
    erzwingen=False,                 # True: auch bereits verarbeitete Bilder neu kodieren/beschreiben
    beschreibung_ins_markdown=False, # True: Beschreibung zusätzlich als Absatz unter das Bild
)
print("pngquant:", CFG.pngquant_binary or "nicht gefunden – Pillow-Paletten als Rückfall")

## 01 — Lauf

Hinweis: Ist das Lektorat noch *unterbrochen*, erst Notebook 02 abschließen – diese Stufe schreibt
`<BUCH>.md` um, danach passte der Lektorat-Checkpoint nicht mehr zur Quelle.

In [ ]:
if lektorat_zustand(BUCH) in ("unterbrochen", "veraltet"):
    raise RuntimeError("Lektorat nicht abgeschlossen – erst Notebook 02 zu Ende laufen lassen.")

ERGEBNIS = bild_lauf.optimiere_dokument(DOCUMENT_DIR, BUCH, CFG)
production = ERGEBNIS.tabelle()
spalten = [c for c in ["picture_ref", "page_no", "status", "technical_class", "selected_candidate",
                       "selected_format", "output_bytes", "saving_ratio", "metadata_status", "alt_text"]
           if c in production.columns]
display(production[spalten])

## 02 — Kontrolle: XMP zurücklesen

In [ ]:
from bild_optimierung.bild_xmp import lies_docrag

zeilen = []
for bild in sorted(pfade.artefakt_ordner(BUCH).glob("*")):
    meta = lies_docrag(bild, CFG)
    zeilen.append({"datei": bild.name, "bytes": bild.stat().st_size,
                   "xmp": bool(meta),
                   "titel": meta["title"] if meta else None,
                   "alt_text": meta["alt_text"] if meta else None,
                   "schlagworte": len(meta["keywords"]) if meta else 0,
                   "klasse": meta["docrag"].get("technicalClass") if meta else None})
display(pd.DataFrame(zeilen))

print("Probleme:", ERGEBNIS.probleme or "keine")
print("Manifest:", ERGEBNIS.manifest_json)


## 11 — Enabling Gemma 4 12B locally

1. Load a vision-capable Gemma 4 12B model in LM Studio.
2. Start the local server from the **Developer** tab.
3. Ensure it is visible through `GET http://localhost:1234/v1/models`.
4. Set `ENABLE_LLM = True`.
5. If necessary, set `LM_STUDIO_MODEL` to the exact model identifier returned by `/v1/models`.
6. Re-run the notebook.

If LM Studio is unavailable, raster optimization still completes. Deterministic Docling provenance is embedded into XMP and the manifest records `SEMANTIC_METADATA_PENDING`. This permits a later metadata-only retry without another lossy raster encoding.

Before archive-wide deployment, replace `urn:docrag:metadata:1.0` with an organisation-owned namespace URI and independently inspect sample files with current ExifTool.



## v0.5 operational note

For best palette quality, install current `pngquant` and verify that `PNGQUANT_BINARY` resolves to an executable path. If `pngquant` is unavailable, the notebook automatically falls back to Pillow while still enforcing the 16-color minimum and CIEDE2000 color-quality gates.

`COLOR_GRAPHIC_MIN_PALETTE_COLORS = 16` is a hard project policy in this revision.
